

#### INF201 – Deliverable 2

Campus Coffee Shop | Group G4_1




#### Setup – Import libraries

Import the libraries and choose the data folder.


In [2]:
# Import libraries
import json
import re
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

DATA_DIR = Path("data")



---

#### Task 1.1 – Read a log file

Read UTF-8 files. If this fails, try Latin-1.


In [3]:
def read_log_file(file_path: str | Path) -> tuple[str, str]:
    """Read a log file and return its text and the encoding used."""
    file_path = Path(file_path)

    try:
        text = file_path.read_text(encoding="utf-8")
        return text, "utf-8"
    except UnicodeDecodeError:
        text = file_path.read_text(encoding="iso-8859-1")
        return text, "iso-8859-1"



#### Task 1.2 – Parse log lines with regular expressions

Extract six fields from valid sales lines. Ignore system messages.


In [4]:
def parse_log(text: str) -> list[dict[str, str | float]]:
    """Parse sales transactions from log text, ignoring unrelated lines."""
    pattern = (
        r"\[(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})\] "
        r"\[(POS_\d+)\] "
        r"item=([A-Za-z ]+?) price=(\d+\.\d{2}) "
        r"(EUR|NOK|USD) "
        r"\((SUCCESS|REFUND|CANCELLED)\)"
    )

    transactions: list[dict[str, str | float]] = []

    for line in text.splitlines():
        match = re.fullmatch(pattern, line.strip())
        if match:
            timestamp, register, item, price, currency, status = match.groups()
            transactions.append({
                "timestamp": timestamp,
                "register_id": register,
                "item": item,
                "price": float(price),
                "currency": currency,
                "status": status,
            })

    return transactions



#### Task 1 – Quick test

Check that the parser ignores a warning line.


In [5]:
example = "[2026-10-15 08:30:00] [POS_01] item=Latte price=4.50 EUR (SUCCESS)\n--- PRINTER PAPER LOW ---"
print(parse_log(example))

[{'timestamp': '2026-10-15 08:30:00', 'register_id': 'POS_01', 'item': 'Latte', 'price': 4.5, 'currency': 'EUR', 'status': 'SUCCESS'}]




---

#### Task 2.1 – Find .log files recursively

Search folders and subfolders 

In [6]:
def find_log_files(directory: str | Path) -> list[Path]:
    """Find all .log files in a directory and its subdirectories."""
    directory = Path(directory)
    log_files: list[Path] = []

    for item in directory.iterdir():
        if item.is_dir():
            # Search inside this subdirectory by calling this function again.
            log_files.extend(find_log_files(item))
        elif item.is_file() and item.suffix == ".log":
            log_files.append(item)

    return log_files



#### Task 2.2 – Find active register IDs

Search nested dictionaries and lists.


In [7]:
def find_active_registers(data: Any) -> set[str]:
    """Find active register IDs in nested dictionaries and lists."""
    active: set[str] = set()

    if isinstance(data, dict):
        register_id = data.get("id")
        if data.get("active") is True and isinstance(register_id, str):
            active.add(register_id)

        for value in data.values():
            if isinstance(value, (dict, list)):
                active.update(find_active_registers(value))

    elif isinstance(data, list):
        for value in data:
            if isinstance(value, (dict, list)):
                active.update(find_active_registers(value))

    return active



#### Task 2 – Quick test

Check the number of files and active registers.


In [8]:
log_files = find_log_files(DATA_DIR)
with (DATA_DIR / "store_config.json").open(encoding="utf-8") as file:
    config = json.load(file)
active_registers = find_active_registers(config)
print("Log files:", len(log_files))
print("Active registers:", sorted(active_registers))

Log files: 9
Active registers: ['POS_01', 'POS_02', 'POS_03']




---

#### Task 3.1 – Remove price outliers

Replace prices more than 3 standard deviations from the mean.


In [9]:
def remove_price_outliers(prices: np.ndarray | list[float]) -> np.ndarray:
    """Replace prices more than three standard deviations from the mean.

    Args:
        prices: Price values as a NumPy array or a list of numbers.

    Returns:
        A one-dimensional float array with outliers replaced by the median
        of the valid prices.
    """
    price_array = np.array(prices, dtype=float).reshape(-1)

    if price_array.size == 0:
        return price_array

    mean_price = np.mean(price_array)
    standard_deviation = np.std(price_array)
    outlier_mask = np.abs(price_array - mean_price) > 3 * standard_deviation

    valid_prices = price_array[~outlier_mask]
    if valid_prices.size > 0:
        price_array[outlier_mask] = np.median(valid_prices)

    return price_array



#### Task 3.2 – Calculate refund ratios

Divide refund counts by sales counts plus 0.000001.


In [10]:
def compute_refund_ratio(
    sales_count: np.ndarray,
    refund_count: np.ndarray,
) -> np.ndarray:
    """Calculate refund ratios while avoiding division by zero.

    Args:
        sales_count: Number of successful sales for each item.
        refund_count: Number of refunds for each item.

    Returns:
        An array of refund ratios, calculated as refunds divided by sales
        plus a small value to avoid division by zero.
    """
    sales = np.asarray(sales_count, dtype=float)
    refunds = np.asarray(refund_count, dtype=float)

    return refunds / (sales + 1e-6)



#### Task 3 – Quick test

Check the price cleaning and refund ratio functions.


In [11]:
print("Prices:", remove_price_outliers([4.5] * 20 + [999.0]))
print("Ratios:", compute_refund_ratio(np.array([10, 0]), np.array([2, 0])))

Prices: [4.5 4.5 4.5 4.5 4.5 4.5 4.5 4.5 4.5 4.5 4.5 4.5 4.5 4.5 4.5 4.5 4.5 4.5
 4.5 4.5 4.5]
Ratios: [0.19999998 0.        ]




---

#### Task 4.1 – Create a DataFrame

Convert timestamps and sort the transactions.


In [12]:
def create_dataframe(transactions: list[dict]) -> pd.DataFrame:
    # Create a table with timestamps as the index
    df = pd.DataFrame(transactions)
    # Convert timestamps to dates
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df = df.set_index("timestamp")
    return df.sort_index()



#### Task 4.2 – Calculate hourly revenue

Sum prices for each item in 1-hour windows.


In [13]:
def hourly_revenue(df: pd.DataFrame) -> pd.DataFrame:
    # Sum sales by item and hour
    # Sum prices per item and hour.
    return df.groupby("item").resample("1h")["price"].sum().reset_index()



#### Task 4.3 – Create a status pivot table

Sum prices by item and transaction status.


In [14]:
def status_pivot_table(df: pd.DataFrame) -> pd.DataFrame:
    # Sum prices by item and status
    # Show prices grouped by item and status.
    pivot = df.pivot_table(
        index="item",
        columns="status",
        values="price",
        aggfunc="sum",
        fill_value=0,
    )
    return pivot.reindex(
        columns=["SUCCESS", "REFUND", "CANCELLED"], fill_value=0
    )



#### Task 4.4 – Calculate refund ratios per item

Count SUCCESS and REFUND transactions for each item.


In [15]:
def item_refund_ratios(df: pd.DataFrame) -> pd.DataFrame:
    # Calculate refund ratios for each item

    # Count sales and refunds
    counts = pd.crosstab(df["item"].to_numpy(), df["status"].to_numpy())

    # Keep the two status columns
    counts = counts.reindex(columns=["SUCCESS", "REFUND"], fill_value=0)

    # Convert counts to NumPy arrays
    sales = counts["SUCCESS"].to_numpy()
    refunds = counts["REFUND"].to_numpy()

    # Calculate refund ratios
    counts["refund_ratio"] = compute_refund_ratio(sales, refunds)

    return counts.rename(columns={
        "SUCCESS": "sales_count",
        "REFUND": "refund_count",
    })



---

#### Task 5 – Full workflow






#### Task 5.1 – Find files and active registers

Load the JSON configuration and find the log files.


In [16]:
# Load store settings
with (DATA_DIR / "store_config.json").open(encoding="utf-8") as file:
    config = json.load(file)

active_registers = find_active_registers(config)
log_files = find_log_files(DATA_DIR)
print("Active registers:", sorted(active_registers))
print("Log files:", len(log_files))

Active registers: ['POS_01', 'POS_02', 'POS_03']
Log files: 9




#### Task 5.2 – Read and parse the logs

Keep transactions from active registers.


In [17]:
# Read each log file
transactions = []
for file_path in log_files:
    text, encoding = read_log_file(file_path)
    for record in parse_log(text):
        # Keep only active registers
        if record["register_id"] in active_registers:
            transactions.append(record)

print("Transactions:", len(transactions))

Transactions: 720




#### Task 5.3 – Build the DataFrame

Show the first rows of the transaction table.


In [18]:
# Preview transactions
df = create_dataframe(transactions)
display(df.head())

,register_id,item,price,currency,status
timestamp,,,,,
2026-10-15 07:00:21,POS_02,Latte,4.47,NOK,SUCCESS
2026-10-15 07:00:22,POS_03,Muffin,3.67,EUR,SUCCESS
2026-10-15 07:00:25,POS_01,Latte,4.10,EUR,SUCCESS
2026-10-15 07:00:48,POS_03,Latte,4.91,USD,SUCCESS
2026-10-15 07:01:28,POS_02,Cappuccino,3.95,USD,SUCCESS




#### Task 5.4 – Clean price outliers

Remove extreme prices before correcting NOK.


In [19]:
# Replace unusual prices
original_prices = df["price"].to_numpy(copy=True)
clean_prices = remove_price_outliers(original_prices)
# Update the price column
df["price"] = clean_prices

print("Outliers replaced:", np.count_nonzero(original_prices != clean_prices))

Outliers replaced: 24




#### Task 5.5 – Correct NOK prices

Multiply NOK prices by 10.


In [20]:
# Correct NOK values
# Find transactions in NOK
is_nok = df["currency"] == "NOK"
df.loc[is_nok, "price"] = df.loc[is_nok, "price"] * 10.0
print("NOK prices corrected:", is_nok.sum())

NOK prices corrected: 230




#### Task 5.6 – Generate reports

Create hourly revenue, status totals and refund ratios.


In [21]:
# Generate three reports
hourly = hourly_revenue(df)
pivot = status_pivot_table(df)
refunds = item_refund_ratios(df)

display(hourly.head())
display(pivot)
display(refunds)

,item,timestamp,price
0,Cappuccino,2026-10-15 07:00:00,1020.20
1,Cappuccino,2026-10-15 08:00:00,1149.75
2,Cappuccino,2026-10-15 09:00:00,177.43
3,Croissant,2026-10-15 07:00:00,919.20
4,Croissant,2026-10-15 08:00:00,831.41


status,SUCCESS,REFUND,CANCELLED
item,,,
Cappuccino,1575.64,314.28,457.46
Croissant,1100.99,453.95,315.04
Espresso,686.74,214.38,275.53
Latte,1259.97,239.01,261.36
Muffin,1042.25,295.94,272.33
Sandwich,2329.56,681.26,492.38


col_0,sales_count,refund_count,refund_ratio
row_0,,,
Cappuccino,93,21,0.225806
Croissant,71,28,0.394366
Espresso,63,18,0.285714
Latte,86,16,0.186047
Muffin,90,20,0.222222
Sandwich,82,20,0.243902




#### Task 5.7 – Save CSV reports

Save the cleaned transactions and two summary tables.


In [22]:
# Save CSV files
df.to_csv("cleaned_transactions.csv")
pivot.to_csv("status_pivot_table.csv")
refunds.to_csv("item_refund_ratios.csv")
print("Saved three CSV files.")

Saved three CSV files.




#### Task 5.7 – Save the JSON report

Save hourly revenue as JSON.


In [23]:
# Save hourly totals as JSON
# Group hourly totals by item
hourly_report = {}
for item, group in hourly.groupby("item"):
    hourly_report[item] = {}
    for _, row in group.iterrows():
        time = row["timestamp"].strftime("%Y-%m-%d %H:%M:%S")
        hourly_report[item][time] = float(row["price"])

with Path("hourly_revenue.json").open("w", encoding="utf-8") as file:
    json.dump(hourly_report, file, indent=2)

print("Saved hourly_revenue.json")

Saved hourly_revenue.json


#### Discussion of results

The results show that the program processed 720 transactions from 3 active registers across 9 log files.

A total of 24 price outliers were found, showing that some transactions had unusual prices. In addition, 230 NOK prices were corrected, indicating that several transactions needed adjustments.

The hourly revenue results show how sales varied throughout the day, while the refund ratios help identify how much of the transaction activity involved refunds.

Overall, the results show that data cleaning is important for producing reliable sales reports and understanding transaction patterns.



#### GenAI use – 
During this assignment, we used AI tools to help us understand parts of the Python code

and improve the presentation. The AI tools were useful for explaining

different approaches to solving problems and helping us understand how certain
functions work.
We also used AI tools to get suggestions for improving our code and solving some

programming errors. We reviewed the suggestions and made sure they were relevant to
the assignment

#### GitHub Repository
Our GitHub repository:
https://github.com/minibergset/git-practice